# Scenario 5: The Deep Dive: A Gentle Introduction to Neural Networks

Let's step into the world of deep learning! We'll build a simple Neural Network using TensorFlow and Keras to tackle our classification problem. We'll also learn to monitor the training process and prevent overfitting.

## 1. Import Libraries and Prepare Data

We'll need the `tensorflow` library for this. Feature scaling is also very important for neural networks.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# --- Data Prep --- 
column_names = [
    'id', 'clump_thickness', 'unif_cell_size', 'unif_cell_shape',
    'marg_adhesion', 'single_epith_cell_size', 'bare_nuclei',
    'bland_chrom', 'norm_nucleoli', 'mitoses', 'class'
]
df = pd.read_csv('../breast-cancer-wisconsin.data', names=column_names)
df['bare_nuclei'] = df['bare_nuclei'].replace('?', np.nan)
df['bare_nuclei'] = pd.to_numeric(df['bare_nuclei'])
df['bare_nuclei'].fillna(df['bare_nuclei'].median(), inplace=True)
df.drop('id', axis=1, inplace=True)
df['class'] = df['class'].map({2: 0, 4: 1})
X = df.drop('class', axis=1)
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 2. Build the Neural Network Model

We'll create a `Sequential` model, which is a simple stack of layers. Our model will have:
- An input layer
- Two hidden layers with the `relu` activation function
- A `Dropout` layer for regularization
- An output layer with a `sigmoid` activation function for our binary classification task.

In [ ]:
# HINT: Use `Sequential` to create the model. Add `Dense` layers for the neurons.
# The first layer needs an `input_shape` argument: `(X_train.shape[1],)`

# YOUR CODE HERE
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Print a summary of the model's architecture
model.summary()

## 3. Compile the Model

Before we can train the model, we need to configure the learning process. We do this with the `compile` method, where we specify the optimizer, the loss function, and any metrics we want to track.

In [ ]:
# HINT: For binary classification, a good choice is optimizer='adam', loss='binary_crossentropy', and metrics=['accuracy'].

# YOUR CODE HERE
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 4. Train the Model with Early Stopping

We'll train the model using the `.fit()` method. We'll also use an `EarlyStopping` callback to monitor the validation loss and stop training automatically when the model stops improving, which helps prevent overfitting.

In [ ]:
# HINT: Create an EarlyStopping callback monitoring 'val_loss'.
# Then, call model.fit(), passing in the training data, validation_split=0.2, epochs, and the callback.

# YOUR CODE HERE
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

## 5. Visualize Training History

Plotting the training and validation loss/accuracy over epochs is a great way to see how your model learned and whether it overfitted.

In [ ]:
history_df = pd.DataFrame(history.history)

# Plot Loss
history_df.loc[:, ['loss', 'val_loss']].plot(title="Loss Curve")
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

# Plot Accuracy
history_df.loc[:, ['accuracy', 'val_accuracy']].plot(title="Accuracy Curve")
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.show()

## 6. Evaluate the Final Model

Finally, let's evaluate our trained model on the unseen test set.

In [ ]:
# HINT: Use model.predict() on the test set. Since the output is a probability, you'll need to round it to get the final class (0 or 1).

# YOUR CODE HERE
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype("int32")

print("\n--- Neural Network Evaluation on Test Set ---")
print(classification_report(y_test, y_pred))